# Biomedical Literature Search with Hugging Face + SQL

**Portfolio project:** semantic search, zero-shot classification, SQL data workflows, and quantitative model evaluation.

## What this demonstrates

This notebook builds a small end-to-end data science workflow for biomedical literature:

1. **Ingest** a labeled corpus of neuroscience / pediatric-care abstracts.
2. **Organize and query** the corpus with SQLite and SQL.
3. Generate **dense text embeddings** with a pretrained Hugging Face Sentence Transformer.
4. Perform **semantic search** using cosine similarity.
5. **Evaluate retrieval** with Precision@K and Mean Reciprocal Rank (MRR).
6. Apply a Hugging Face **zero-shot transformer classifier**.
7. **Evaluate classification** with accuracy, macro-F1, and a confusion matrix.
8. Persist model outputs back to SQL for downstream analysis.

The notebook includes a small synthetic corpus so the workflow is easy to inspect. Set `USE_LIVE_PUBMED = True` to replace it with abstracts retrieved from PubMed.

> **Why this project?** In industry, using a pretrained model is only part of the job. The more important skill is building a reproducible data pipeline around it and evaluating whether the model is actually useful.

## 1. Environment

Recommended Python: **3.10+**

Install dependencies with:

```bash
pip install -r requirements.txt
```

The first model run downloads pretrained weights from Hugging Face.

In [ ]:
from pathlib import Path
import sqlite3
import time
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score, f1_score, ConfusionMatrixDisplay
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from transformers import pipeline

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DB_PATH = Path("neuroscience_literature.db")
USE_LIVE_PUBMED = False      # Change to True for real PubMed abstracts.
PUBMED_PER_TOPIC = 15        # Used only in live mode.

## 2. Define the analytical question

We will organize papers into four broad domains:

- **myelin_connectomics**
- **eeg_memory**
- **neurodevelopment**
- **pediatric_complex_care**

These labels are used as *weak ground truth* for evaluation. In live PubMed mode, the label reflects the query that retrieved the paper, so it is imperfect: a paper can legitimately span multiple topics.

That imperfection is intentional. A real data scientist should state what the labels mean and what they do **not** mean.

In [ ]:
TOPICS = {
    "myelin_connectomics": {
        "query": '("myelin"[Title/Abstract]) AND (connectome[Title/Abstract] OR connectivity[Title/Abstract])',
        "semantic_query": "How does white matter myelin influence communication and connectivity between brain regions?",
        "display_label": "myelin and brain connectivity",
    },
    "eeg_memory": {
        "query": '(EEG[Title/Abstract] OR electrophysiology[Title/Abstract]) AND memory[Title/Abstract]',
        "semantic_query": "EEG signals and electrophysiological mechanisms related to human memory",
        "display_label": "EEG and memory",
    },
    "neurodevelopment": {
        "query": '(neurodevelopment[Title/Abstract] OR developmental[Title/Abstract]) AND (brain[Title/Abstract] OR MRI[Title/Abstract])',
        "semantic_query": "brain development, neurodevelopment, and developmental neuroimaging",
        "display_label": "neurodevelopment",
    },
    "pediatric_complex_care": {
        "query": 'pediatric[Title/Abstract] AND ("complex care"[Title/Abstract] OR "care coordination"[Title/Abstract])',
        "semantic_query": "care coordination and outcomes for children with complex medical needs",
        "display_label": "pediatric complex care",
    },
}

LABEL_TO_TOPIC = {v["display_label"]: k for k, v in TOPICS.items()}

## 3. Data ingestion

### Reproducible demo mode

The built-in records below are **synthetic summaries**, not copied abstracts. They make the notebook inspectable and deterministic.

### Live mode

If `USE_LIVE_PUBMED=True`, the notebook uses NCBI E-utilities to retrieve current PubMed records for each topic and parses titles, abstracts, publication years, and PubMed IDs.

In [ ]:
SYNTHETIC_PAPERS = [
    ("M001", "Myelin-sensitive MRI and network communication",
     "We examine whether regional variation in white-matter myelin is associated with communication efficiency across structural brain networks. Myelin-sensitive measurements are integrated with tractography-derived connectivity and graph communication models.",
     2025, "myelin_connectomics"),
    ("M002", "White-matter microstructure and functional coupling",
     "This study evaluates relationships between tract microstructure, conduction-related features, and functional connectivity. Network models are compared across spatial scales to identify robust structure-function associations.",
     2024, "myelin_connectomics"),
    ("M003", "Tract-specific myelin and macroscale connectivity",
     "Quantitative MRI measurements of tract-specific myelin are combined with structural connectivity to test whether microstructural variation improves prediction of functional network organization.",
     2023, "myelin_connectomics"),
    ("M004", "Communication models in weighted connectomes",
     "Multiple routing and diffusion communication models are evaluated in structural connectomes weighted by biological features of white matter. Model behavior is compared using edgewise and regional outcomes.",
     2022, "myelin_connectomics"),
    ("M005", "Conduction properties and brain network dynamics",
     "We investigate how heterogeneity in axonal and myelin properties may alter effective communication across large-scale brain networks and contribute to regional differences in functional coupling.",
     2021, "myelin_connectomics"),

    ("E001", "Event-related potentials predict later memory",
     "Electroencephalographic responses during encoding are analyzed to determine whether event-related potential features predict subsequent memory performance across participants.",
     2024, "eeg_memory"),
    ("E002", "Oscillatory EEG markers of working memory",
     "Spectral features in theta and alpha frequency bands are evaluated as predictors of working-memory performance using multivariate models and cross-validation.",
     2023, "eeg_memory"),
    ("E003", "P300 dynamics during memory formation",
     "The amplitude and latency of the P300 event-related potential are related to behavioral measures of memory encoding and recall across experimental conditions.",
     2022, "eeg_memory"),
    ("E004", "Multivariate electrophysiology of episodic recall",
     "High-dimensional EEG features are reduced using latent-variable methods and used to characterize individual variation in episodic memory performance.",
     2021, "eeg_memory"),
    ("E005", "Temporal neural signatures of successful encoding",
     "Time-resolved scalp electrophysiology is used to identify neural patterns that distinguish subsequently remembered from forgotten stimuli.",
     2020, "eeg_memory"),

    ("N001", "Development of association cortex networks",
     "Longitudinal neuroimaging is used to characterize maturation of association cortex connectivity from childhood through adolescence, with emphasis on distributed network organization.",
     2025, "neurodevelopment"),
    ("N002", "White-matter maturation across childhood",
     "Diffusion and quantitative MRI measurements are combined to examine developmental trajectories of white-matter microstructure across major pathways.",
     2024, "neurodevelopment"),
    ("N003", "Functional network development in adolescence",
     "Resting-state functional MRI is analyzed across age to quantify developmental changes in segregation and integration of large-scale brain systems.",
     2023, "neurodevelopment"),
    ("N004", "Neurodevelopmental trajectories from multimodal MRI",
     "Structural, diffusion, and functional MRI features are integrated to model age-related changes in brain organization during development.",
     2022, "neurodevelopment"),
    ("N005", "Individual variation in developmental connectomics",
     "Graph-based measures are used to evaluate individual differences in structural and functional network maturation during childhood.",
     2021, "neurodevelopment"),

    ("P001", "Care coordination for children with medical complexity",
     "Healthcare utilization and care-coordination measures are analyzed to identify factors associated with fewer emergency visits and improved continuity for children with complex medical needs.",
     2025, "pediatric_complex_care"),
    ("P002", "Predicting pediatric acute-care utilization",
     "Clinical and administrative features are used to evaluate risk models for emergency and inpatient utilization among medically complex pediatric populations.",
     2024, "pediatric_complex_care"),
    ("P003", "Social determinants in pediatric complex care",
     "Clinical outcomes are modeled together with social and access-related factors to identify barriers affecting children who require coordinated specialty care.",
     2023, "pediatric_complex_care"),
    ("P004", "Home-based care and pediatric outcomes",
     "A retrospective analysis evaluates whether coordinated home-based services are associated with changes in hospital utilization and family-centered outcomes.",
     2022, "pediatric_complex_care"),
    ("P005", "Integrating claims and clinical data for pediatric care",
     "Claims, encounter, and clinical data are integrated to construct longitudinal patient-level features for population health analysis in children with complex conditions.",
     2021, "pediatric_complex_care"),
]

def synthetic_corpus():
    return pd.DataFrame(
        SYNTHETIC_PAPERS,
        columns=["paper_id", "title", "abstract", "year", "source_topic"]
    )

def fetch_pubmed_ids(query, retmax=15):
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    params = {
        "db": "pubmed",
        "term": query,
        "retmode": "json",
        "retmax": retmax,
        "sort": "pub date",
        "tool": "biomedical-literature-portfolio-demo",
    }
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    return response.json()["esearchresult"]["idlist"]

def fetch_pubmed_records(pmids):
    if not pmids:
        return []
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    params = {
        "db": "pubmed",
        "id": ",".join(pmids),
        "retmode": "xml",
        "tool": "biomedical-literature-portfolio-demo",
    }
    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()

    root = ET.fromstring(response.content)
    records = []

    for article in root.findall(".//PubmedArticle"):
        pmid = article.findtext(".//PMID")
        title_node = article.find(".//ArticleTitle")
        title = "".join(title_node.itertext()) if title_node is not None else ""

        abstract_parts = []
        for node in article.findall(".//Abstract/AbstractText"):
            label = node.attrib.get("Label")
            text = "".join(node.itertext()).strip()
            if text:
                abstract_parts.append(f"{label}: {text}" if label else text)
        abstract = " ".join(abstract_parts)

        year = None
        for path in [".//ArticleDate/Year", ".//PubDate/Year", ".//PubDate/MedlineDate"]:
            value = article.findtext(path)
            if value:
                digits = "".join(ch for ch in value[:4] if ch.isdigit())
                if len(digits) == 4:
                    year = int(digits)
                    break

        if title and abstract:
            records.append({
                "paper_id": pmid,
                "title": title,
                "abstract": abstract,
                "year": year,
            })

    return records

def live_pubmed_corpus(per_topic=15):
    all_records = []
    seen_pmids = set()

    for topic, config in TOPICS.items():
        pmids = fetch_pubmed_ids(config["query"], retmax=per_topic)
        records = fetch_pubmed_records(pmids)

        for record in records:
            if record["paper_id"] in seen_pmids:
                continue
            seen_pmids.add(record["paper_id"])
            record["source_topic"] = topic
            all_records.append(record)

        time.sleep(0.4)

    return pd.DataFrame(all_records)

papers = live_pubmed_corpus(PUBMED_PER_TOPIC) if USE_LIVE_PUBMED else synthetic_corpus()
papers["text"] = papers["title"] + ". " + papers["abstract"]

print(f"Corpus size: {len(papers)} papers")
display(papers.head())
display(papers.groupby("source_topic").size().rename("n_papers"))

## 4. Organize the corpus with SQL

SQLite is used here because it is portable and requires no server. The SQL concepts transfer directly to PostgreSQL, Snowflake, BigQuery, and other relational systems.

This section demonstrates:

- schema creation
- inserts
- `SELECT` / `WHERE`
- aggregation
- common table expressions (CTEs)
- window functions
- joins

In [ ]:
if DB_PATH.exists():
    DB_PATH.unlink()

with sqlite3.connect(DB_PATH) as conn:
    conn.execute("""
        CREATE TABLE papers (
            paper_id TEXT PRIMARY KEY,
            title TEXT NOT NULL,
            abstract TEXT NOT NULL,
            year INTEGER,
            source_topic TEXT NOT NULL
        )
    """)

    papers[["paper_id", "title", "abstract", "year", "source_topic"]].to_sql(
        "papers", conn, if_exists="append", index=False
    )

    conn.execute("""
        CREATE TABLE topics (
            source_topic TEXT PRIMARY KEY,
            display_label TEXT NOT NULL,
            semantic_query TEXT NOT NULL
        )
    """)

    topic_rows = pd.DataFrame([
        {
            "source_topic": topic,
            "display_label": config["display_label"],
            "semantic_query": config["semantic_query"],
        }
        for topic, config in TOPICS.items()
    ])
    topic_rows.to_sql("topics", conn, if_exists="append", index=False)

print(f"Created SQLite database: {DB_PATH.resolve()}")

### SQL example: aggregation + CTE + window function

The query below reports topic-level counts and ranks the newest papers within each topic.

In [ ]:
sql = """
WITH ranked_papers AS (
    SELECT
        p.paper_id,
        p.title,
        p.year,
        p.source_topic,
        t.display_label,
        ROW_NUMBER() OVER (
            PARTITION BY p.source_topic
            ORDER BY p.year DESC, p.paper_id
        ) AS recency_rank
    FROM papers AS p
    INNER JOIN topics AS t
        ON p.source_topic = t.source_topic
),
topic_counts AS (
    SELECT
        source_topic,
        COUNT(*) AS n_papers
    FROM papers
    GROUP BY source_topic
)
SELECT
    r.display_label,
    c.n_papers,
    r.paper_id,
    r.title,
    r.year
FROM ranked_papers AS r
INNER JOIN topic_counts AS c
    ON r.source_topic = c.source_topic
WHERE r.recency_rank <= 2
ORDER BY r.display_label, r.recency_rank;
"""

with sqlite3.connect(DB_PATH) as conn:
    sql_preview = pd.read_sql_query(sql, conn)

sql_preview

## 5. Generate embeddings with a Hugging Face pretrained model

We use `sentence-transformers/all-MiniLM-L6-v2`, a pretrained Sentence Transformer available on the Hugging Face Hub.

The model maps text into dense vectors. Semantically related text should lie closer together in embedding space, enabling retrieval without exact keyword matching.

In [ ]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

paper_embeddings = embedding_model.encode(
    papers["text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True,
)

print("Embedding matrix shape:", paper_embeddings.shape)

## 6. Semantic search

A user query is embedded with the same model. We rank papers by cosine similarity between the query embedding and document embeddings.

In [ ]:
def semantic_search(query, top_k=5):
    query_embedding = embedding_model.encode([query], normalize_embeddings=True)
    scores = cosine_similarity(query_embedding, paper_embeddings)[0]
    ranking = np.argsort(scores)[::-1][:top_k]

    results = papers.iloc[ranking][
        ["paper_id", "title", "year", "source_topic"]
    ].copy()
    results["similarity"] = scores[ranking]
    results["rank"] = np.arange(1, len(results) + 1)
    results["query"] = query
    return results.reset_index(drop=True)

example_query = "How does white matter biology affect communication between distant brain regions?"
semantic_search(example_query, top_k=5)

## 7. Evaluate semantic retrieval

Instead of stopping after a plausible-looking search result, we quantify performance.

For each topic, its `semantic_query` is used as a retrieval query. Papers retrieved from the matching source topic are treated as relevant.

We report:

- **Precision@K**: fraction of the top K results that match the expected topic.
- **Reciprocal rank**: inverse rank of the first relevant result.
- **MRR**: mean reciprocal rank across all queries.

Because the topic labels are weak labels rather than expert annotations, these metrics evaluate the demonstration pipeline—not clinical or scientific validity.

In [ ]:
def evaluate_retrieval(k=5):
    rows = []

    for expected_topic, config in TOPICS.items():
        results = semantic_search(config["semantic_query"], top_k=k)
        relevant = results["source_topic"].eq(expected_topic)

        precision_at_k = relevant.mean()
        relevant_ranks = results.loc[relevant, "rank"].tolist()
        reciprocal_rank = 1.0 / min(relevant_ranks) if relevant_ranks else 0.0

        rows.append({
            "expected_topic": expected_topic,
            "query": config["semantic_query"],
            f"precision@{k}": precision_at_k,
            "reciprocal_rank": reciprocal_rank,
        })

    return pd.DataFrame(rows)

retrieval_eval = evaluate_retrieval(k=5)
display(retrieval_eval)

print("Mean Precision@5:", round(retrieval_eval["precision@5"].mean(), 3))
print("Mean Reciprocal Rank:", round(retrieval_eval["reciprocal_rank"].mean(), 3))

## 8. Persist search outputs to SQL

Model outputs become data products in their own right. Here we save ranked search results to SQLite and query them with SQL.

In [ ]:
all_search_results = []

for expected_topic, config in TOPICS.items():
    result = semantic_search(config["semantic_query"], top_k=5)
    result["expected_topic"] = expected_topic
    all_search_results.append(result)

all_search_results = pd.concat(all_search_results, ignore_index=True)

with sqlite3.connect(DB_PATH) as conn:
    all_search_results.to_sql(
        "semantic_search_results",
        conn,
        if_exists="replace",
        index=False,
    )

    joined_results = pd.read_sql_query("""
        SELECT
            s.expected_topic,
            s.rank,
            s.similarity,
            p.paper_id,
            p.title,
            p.year,
            p.source_topic
        FROM semantic_search_results AS s
        INNER JOIN papers AS p
            ON s.paper_id = p.paper_id
        WHERE s.rank <= 3
        ORDER BY s.expected_topic, s.rank;
    """, conn)

joined_results

## 9. Zero-shot classification with Hugging Face Transformers

A zero-shot classifier predicts among labels supplied at inference time; the model was not specifically trained on our four project categories.

This is useful when labeled training data are scarce, but it makes **evaluation essential**.

In [ ]:
ZERO_SHOT_MODEL = "MoritzLaurer/DeBERTa-v3-base-mnli"

zero_shot = pipeline(
    task="zero-shot-classification",
    model=ZERO_SHOT_MODEL,
)

candidate_labels = [TOPICS[t]["display_label"] for t in TOPICS]
candidate_labels

To keep the demo reasonably fast on CPU, live mode evaluates at most 24 papers. The built-in synthetic corpus is small enough to evaluate in full.

In [ ]:
def stratified_sample(frame, max_n=24):
    if len(frame) <= max_n:
        return frame.copy()

    pieces = []
    topics = frame["source_topic"].unique()
    per_topic = max(1, max_n // len(topics))

    for topic in topics:
        subset = frame[frame["source_topic"] == topic]
        pieces.append(
            subset.sample(
                n=min(per_topic, len(subset)),
                random_state=RANDOM_STATE
            )
        )

    return pd.concat(pieces, ignore_index=True).head(max_n)

classification_set = stratified_sample(papers, max_n=24)
print("Papers selected for zero-shot evaluation:", len(classification_set))

In [ ]:
predictions = []

for row in classification_set.itertuples(index=False):
    result = zero_shot(
        row.text,
        candidate_labels=candidate_labels,
        multi_label=False,
    )

    predicted_label = result["labels"][0]

    predictions.append({
        "paper_id": row.paper_id,
        "source_topic": row.source_topic,
        "predicted_label": predicted_label,
        "predicted_topic": LABEL_TO_TOPIC[predicted_label],
        "confidence": result["scores"][0],
    })

classification_results = pd.DataFrame(predictions)
classification_results.head()

## 10. Evaluate zero-shot classification

We evaluate:

- **Accuracy**
- **Macro-F1**, which weights each class equally
- **Confusion matrix**
- confidence of correct versus incorrect predictions

In [ ]:
y_true = classification_results["source_topic"]
y_pred = classification_results["predicted_topic"]

accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

print(f"Accuracy: {accuracy:.3f}")
print(f"Macro-F1: {macro_f1:.3f}")

In [ ]:
label_order = list(TOPICS.keys())

fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y_true,
    y_pred,
    labels=label_order,
    xticks_rotation=45,
    ax=ax,
)
ax.set_title("Zero-shot classification confusion matrix")
plt.tight_layout()
plt.show()

In [ ]:
classification_results["correct"] = (
    classification_results["source_topic"]
    == classification_results["predicted_topic"]
)

confidence_summary = (
    classification_results
    .groupby("correct")["confidence"]
    .agg(["count", "mean", "median", "min", "max"])
)

confidence_summary

## 11. Persist predictions and inspect failure cases with SQL

This is an important DS habit: don't just report a single score. Inspect where the model fails.

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    classification_results.to_sql(
        "zero_shot_predictions",
        conn,
        if_exists="replace",
        index=False,
    )

    failure_query = """
    WITH prediction_audit AS (
        SELECT
            z.paper_id,
            p.title,
            z.source_topic,
            z.predicted_topic,
            z.confidence,
            CASE
                WHEN z.source_topic = z.predicted_topic THEN 1
                ELSE 0
            END AS correct
        FROM zero_shot_predictions AS z
        INNER JOIN papers AS p
            ON z.paper_id = p.paper_id
    )
    SELECT
        paper_id,
        title,
        source_topic,
        predicted_topic,
        ROUND(confidence, 3) AS confidence
    FROM prediction_audit
    WHERE correct = 0
    ORDER BY confidence DESC;
    """

    failures = pd.read_sql_query(failure_query, conn)

failures

## 12. Optional extension: generative foundation-model synthesis

A natural next step is to add an instruction-tuned generative model, for example a FLAN-T5 model from Hugging Face, to synthesize the top retrieved abstracts.

**Important:** generative summaries should not be treated as ground truth. A stronger project would evaluate them against human-written references or explicit factuality criteria.

Example extension:

```python
from transformers import pipeline

generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-small"
)

top = semantic_search(
    "How does myelin influence brain-network communication?",
    top_k=3
)

context = "\n".join(
    papers.set_index("paper_id").loc[top["paper_id"], "abstract"].tolist()
)

prompt = (
    "Using only the evidence below, provide a two-sentence synthesis "
    "and explicitly state any uncertainty.\n\n"
    + context[:3000]
)

generator(prompt, max_new_tokens=100)
```

This extension adds generative-model use, but the **evaluation plan** is more important than simply producing text.

# Results and limitations

When presenting this project, focus on the workflow rather than claiming domain-level performance.

## Skills demonstrated

- Hugging Face `transformers`
- Sentence Transformers / embeddings
- Semantic search
- Zero-shot classification
- Pretrained / foundation-model workflows
- SQL / SQLite
- CTEs, joins, aggregation, window functions
- pandas / NumPy
- scikit-learn model evaluation
- Precision@K and MRR
- Accuracy, macro-F1, confusion matrices
- Reproducible data pipelines
- Failure analysis

## Important limitations

1. The built-in corpus is synthetic and intentionally small.
2. Live PubMed topic labels are query-derived weak labels, not expert annotations.
3. Semantic relevance is more nuanced than a four-class label.
4. Zero-shot confidence scores should not be interpreted as calibrated probabilities.
5. A production system would require stronger annotation, model comparison, monitoring, privacy/security review, and scalability testing.

## Interview-ready summary

> I built an end-to-end biomedical literature retrieval and classification project using Hugging Face pretrained transformers. I stored and queried the corpus in SQLite, generated sentence embeddings for semantic search, evaluated retrieval with Precision@K and MRR, applied zero-shot classification, and evaluated classification with accuracy, macro-F1, confusion matrices, confidence analysis, and explicit failure-case review. The point of the project was not just to call a foundation model, but to build and evaluate a reproducible data workflow around it.